# Optimizing LibreYOLO for Qualcomm® Snapdragon Devices Using QAIRT

This notebook walks through the full pipeline to optimize a **LibreYOLO** object detection model for efficient inference on Qualcomm® Snapdragon devices using the **Qualcomm AI Runtime (QAIRT) SDK**.

The pipeline covers:
1. **Image preprocessing** — resize and normalize images for model input.
2. **Calibration dataset preparation** — generate representative `.raw` samples for INT8 quantization.
3. **Model export** — load a pre-trained LibreYOLOXs checkpoint and export it to ONNX.
4. **DLC conversion** — convert the ONNX model to QAIRT's Deep Learning Container (DLC) format using `qairt-converter`.
5. **INT8 quantization** — apply post-training quantization for faster, more efficient inference using `qairt-quantizer`.
6. **HTP context binary compilation** — compile the DLC offline for the Hexagon Tensor Processor (HTP) on the Snapdragon 778G (sm7325) using `qairt-context-binary-generator`.

We begin by importing the necessary libraries.

In [1]:
# Import necessary libraries.
import cv2
import glob
import os
import onnx
import random
import torch
import uuid

import numpy as np

from libreyolo.models.yolox.nn import LibreYOLOXModel

## Preprocessing and Calibration Data

Before converting or quantizing the model, two things are needed:

- A **preprocessing function** to transform raw images into the tensor format expected by LibreYOLO (letterbox-resized to 640×640, BGR color, 0–255 float32, CHW layout).
- A **representative calibration dataset** in `.raw` binary format. QAIRT's `qairt-quantizer` uses this dataset during post-training quantization to compute per-layer scale factors for INT8 weights and activations.

The cell below defines the `letterbox()` and `preprocess()` helper functions used throughout this pipeline.

In [2]:
def letterbox(image: np.ndarray, target: int = 640, pad_value: int = 114) -> tuple:
    """Resize image to (target×target) with top-left letterbox padding.

    YOLOX preprocessing: image is placed at top-left (0, 0); padding fills
    the right and bottom edges. Color format stays BGR, values stay 0–255.

    Args:
        image (np.ndarray): Input BGR image as NumPy array.
        target (int): Target image size (square).
        pad_value (int): Padding pixel value (default 114, YOLOX convention).

    Returns:
        padded (np.ndarray): Padded image, float32 HWC BGR, 0-255 range.
        ratio (float): Scale factor applied to original dimensions.
    """
    h, w = image.shape[:2]
    ratio = min(target / h, target / w)
    new_w, new_h = int(w * ratio), int(h * ratio)

    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    padded = np.full((target, target, 3), pad_value, dtype=np.float32)
    padded[:new_h, :new_w] = resized

    return padded, ratio


def preprocess(original_image: np.ndarray, size: int = 640) -> np.ndarray:
    """
    Preprocess the input image for model inference.

    YOLOX expects: BGR, 0-255 float32, CHW, top-left letterbox.
    No /255 normalisation and no BGR→RGB conversion.

    Args:
        original_image (np.ndarray): Input BGR image as NumPy array.
        size (int): Target image size.

    Returns:
        np.ndarray: Preprocessed float32 image, CHW layout, 0-255 range.
    """
    padded, _ = letterbox(original_image, size)
    tensor = padded.transpose(2, 0, 1)  # HWC → CHW
    return np.ascontiguousarray(tensor)

### Preparing the Calibration Dataset

QAIRT's quantization tool (`qairt-quantizer`) requires input samples in `.raw` format — flat binary files containing `float32` pixel values with shape `(C, H, W)`.

The code below:
1. Downloads the **COCO 2017 validation set** (~777 MB).
2. Randomly samples **2000 images** (with a fixed seed for reproducibility).
3. Preprocesses each image using `preprocess()` and saves it as a `.raw` file.
4. Generates a `filenames.txt` manifest listing all `.raw` files — required by `qairt-quantizer`.

In [3]:
if not os.path.exists("val"):
    !bash -c 'curl -L -o val2017.zip http://images.cocodataset.org/zips/val2017.zip'
    !bash -c 'unzip val2017.zip -d val'
    !bash -c 'rm val2017.zip'

if not os.path.exists("raw"):
    !bash -c 'mkdir raw'

    random.seed(42)
    SAMPLE_SIZE = 2000

    filenames = glob.glob(f"val/**/*.jpg", recursive=True)
    random.shuffle(filenames)

    for filename in filenames[:SAMPLE_SIZE]:
        original_image = cv2.imread(filename)
        processed_image = preprocess(original_image)
        processed_image.tofile(f"raw/{uuid.uuid4()}.raw")

if not os.path.exists("raw/filenames.txt"):
    !bash -c 'find raw -name "*.raw" > raw/filenames.txt'

### Loading LibreYOLO and Exporting to ONNX

SNPE does not consume LibreYOLO models in PyTorch format directly. The model must first be exported to the **ONNX (Open Neural Network Exchange)** format, which SNPE can then parse and convert to its DLC format.

The code below:
1. Downloads the pre-trained **LibreYOLOXs** checkpoint from Hugging Face.
2. Loads it using the `LibreYOLO` wrapper and sets it to evaluation mode.
3. Exports it to ONNX using `torch.onnx.export()` with `opset_version=18`.

The model decodes the grid offsets inside the graph and returns:
[1, 8400, 85]

where:

8400 = 80x80 + 40x40 + 20x20
85 = 4 bbox values + 1 objectness + 80 class probabilities

The bbox format is:

[cx, cy, w, h]

in the resized 640x640 input coordinate space. When use the model, it is necessary the confidence filtering, `cxcywh -> xyxy` conversion, scaling back to the original image, and NMS.

In [4]:
os.makedirs("../models", exist_ok=True)
os.makedirs("../models/qairt", exist_ok=True)

if not os.path.exists("../models/LibreYOLOXs.pt"):
    !bash -c 'curl -L -o ../models/LibreYOLOXs.pt https://huggingface.co/LibreYOLO/LibreYOLOXs/resolve/main/LibreYOLOXs.pt?download=true'

checkpoint = torch.load(
    "../models/LibreYOLOXs.pt",
    map_location="cpu"
)

model = LibreYOLOXModel(config="s", nb_classes=80)
model.load_state_dict(checkpoint["model"], strict=True)

model.eval()
model.head.export = True

dummy = torch.randn(1, 3, 640, 640)
onnx_path = "../models/LibreYOLOXs.onnx"

if not os.path.exists(onnx_path):
    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        opset_version=13,
        dynamo=False,
        do_constant_folding=True,
        input_names=["images"],
        output_names=["detections"]
    )

    print(f"Exported decoded ONNX to: {onnx_path}.")

### Validating the ONNX output shape

This cell checks that the exported ONNX has exactly one output named `detections` with shape `[1, 8400, 85]`. If you still see three outputs, the model was exported without `torch_model.head.export = True`.

In [5]:
onnx_model = onnx.load("../models/LibreYOLOXs.onnx")
onnx.checker.check_model(onnx_model)

print("Inputs:")
for tensor in onnx_model.graph.input:
    dims = [d.dim_value if d.dim_value > 0 else d.dim_param for d in tensor.type.tensor_type.shape.dim]
    print(f" {tensor.name}: {dims}")

print("Outputs:")
output_shapes = {}
for tensor in onnx_model.graph.output:
    dims = [d.dim_value if d.dim_value > 0 else d.dim_param for d in tensor.type.tensor_type.shape.dim]
    output_shapes[tensor.name] = dims
    print(f" {tensor.name}: {dims}")

assert len(onnx_model.graph.output) == 1, "Expected exactly one decoded output."
assert "detections" in output_shapes, "Expected output named `detections`."
assert output_shapes["detections"] == [1, 8400, 85], f"Unexpected output shape: {output_shapes['detections']}"

print("ONNX export is correct: one decoded output [1, 8400, 85].")

Inputs:
 images: [1, 3, 640, 640]
Outputs:
 detections: [1, 8400, 85]
ONNX export is correct: one decoded output [1, 8400, 85].


## Converting the Model to QAIRT DLC Format

QAIRT uses the **DLC (Deep Learning Container)** format as an intermediate representation. The first step is to convert the ONNX model to a floating-point (**FP32**) DLC using the `qairt-converter` tool. This DLC file is the starting point for all subsequent quantization and compilation steps.

In [6]:
!bash -c 'qairt-converter \
    --input_network ../models/LibreYOLOXs.onnx \
    --output_path ../models/qairt/LibreYOLOXs_fp32.dlc'

2026-05-10 18:20:24,332 - 278 - INFO - INFO_INITIALIZATION_SUCCESS: 
2026-05-10 18:20:24,378 - 283 - WARNING - Unable to register converter supported Operation [Inverse:Version 1] with your Onnx installation. Converter will bail if Model contains this Op.
2026-05-10 18:20:24,460 - 283 - WARNING - --desired_input_shape and -d are deprecated. Use --source_model_input_shape or -s for achieving this functionality
2026-05-10 18:20:24,460 - 278 - INFO - Input shape info 
2026-05-10 18:20:26,003 - 283 - WARNING - Onnx model simplification failed due to: No module named 'onnxsim'
2026-05-10 18:20:26,003 - 283 - WARNING - Simplified model validation failed
2026-05-10 18:20:26,256 - 283 - WARNING - Onnx model simplification failed due to: No module named 'onnxsim'
2026-05-10 18:20:26,256 - 283 - WARNING - Simplified model validation failed
2026-05-10 18:20:26,429 - 278 - INFO - INFO_STATIC_RESHAPE: Applying static reshape to /head/Concat_3_output_0: new name /head/Reshape_1_output_0 new shape [1

### Inspecting the FP32 DLC

The `qairt-dlc-info` command prints a detailed summary of the DLC graph, including layer names, types, input/output tensor shapes, and supported execution backends (CPU, GPU, HTP). Reviewing this output verifies that the ONNX export was clean and that all operators are supported by QAIRT before proceeding to quantization.

In [7]:
!bash -c 'qairt-dlc-info \
    --input_dlc ../models/qairt/LibreYOLOXs_fp32.dlc'

DLC info of: /workspace/models/qairt/LibreYOLOXs_fp32.dlc
                                                                                                    
Model Version: 
Model Copyright:
Converter command: qairt-converter; validate_models=False; use_quantize_v2=False; use_onnx_relay=False; unroll_lstm_time_steps=True; unroll_gru_time_steps=True; transforms_metadata=None; tf_summary=False; skip_validation=False; signature_name=; dump_value_info=False; remove_unused_inputs=False; expand_sparse_op_structure=False; quantizer_log_level=LogLevel.NONE; quantization_overrides=; enable_match_topk=False; pytorch_custom_op_lib=; prepare_inputs_as_params=False; perform_axes_to_spatial_first_order=False; define_symbol=None; use_convert_quantization_nodes=False; packed_max_seq=1; packed_masked_softmax_inputs=[]; output_layout=[]; float_bias_bitwidth=0; show_unconsumed_nodes=False; out_names=[]; float_bitwidth=32; optimization_pass_mode=ir_optimizer_disable; apply_masked_softmax=uncompressed; en

### INT8 Post-Training Quantization

Running inference in **8-bit integer (INT8)** precision is significantly faster and more energy-efficient on Qualcomm® hardware than FP32 — with only a small accuracy trade-off. The `qairt-quantizer` tool applies **post-training quantization (PTQ)** by computing per-layer scale factors from the calibration `.raw` samples prepared earlier, producing an INT8 DLC.

In [8]:
!bash -c 'qairt-quantizer \
    --input_dlc ../models/qairt/LibreYOLOXs_fp32.dlc \
    --input_list raw/filenames.txt \
    --output_dlc ../models/qairt/LibreYOLOXs_int8.dlc'

     0.9ms [  INFO ] Inferences will run in sync mode
Processing inference input(s):
raw/37f50879-f629-4031-aad7-11a6f275b00b.raw
raw/62caa7ce-10f9-4662-a28c-a11c73887828.raw
raw/08acaad7-ed4b-4332-b94a-b0ccda017b95.raw
raw/33d9a2d1-5f90-4434-98a3-3edb1b37cae4.raw
raw/891327c8-2523-4673-ba47-8b723c697992.raw
raw/12c65b90-6e09-4d96-be39-ab35ea592716.raw
raw/c9886ddb-5175-4ba4-a34d-0649697db970.raw
raw/e9f37b32-3873-4b90-8a6d-9005ca580e08.raw
raw/10f275be-4d70-467e-a0da-b19a11f60135.raw
raw/66c0b7d9-8bc1-4212-adf6-c72e92a1a5dd.raw
raw/ec319f5c-4cc9-450a-a5e3-fa395a42e9ef.raw
raw/7874b6aa-8815-46cd-905f-ba93937f0f9e.raw
raw/edebeb51-ea1f-4153-b5b5-490ae33c12b5.raw
raw/4a9c4fa2-7d1c-4d8e-a6dc-b07c2d1efd4d.raw
raw/5c307190-9262-43b0-b9d1-c4297b054fcb.raw
raw/50d4edb6-6fdb-49f5-96f5-6c8607a64503.raw
raw/f939457b-583e-40ac-9d1d-cceea3dd8a04.raw
raw/98950b0a-8e07-4c25-9835-ccdbbeaa00ad.raw
raw/312733ac-cc39-49c1-82f3-8db3fc3ab2c1.raw
raw/1b4d5ff8-40be-4e36-bada-f4cdc38a7186.raw
raw/2aa4e841-2d

### Inspecting the INT8 DLC

After quantization, `qairt-dlc-info` is used again to inspect the INT8 DLC. Compared to the FP32 output, you should observe that layer types now reflect quantized variants. The INT8 DLC is expected to be smaller than the FP32 DLC, but the actual reduction should be measured.

In [9]:
!bash -c 'qairt-dlc-info \
    --input_dlc ../models/qairt/LibreYOLOXs_int8.dlc'

DLC info of: /workspace/models/qairt/LibreYOLOXs_int8.dlc
                                                                                                    
Model Version: 
Model Copyright:
Converter command: qairt-converter; optimization_pass_mode_config=; batch=None; squash_box_decoder=True; dump_exported_onnx=False; dumpIR=False; custom_op_config_paths=None; preprocess_roi_pool_inputs=True; saved_model_tag=serve; backend=HTP; preserve_io=[['layout']]; multi_time_steps_lstm=False; debug=-1; calc_static_encodings=False; dump_custom_io_config_template=; partial_graph_input_name=None; perform_sequence_construct_optimizer=False; adjust_nms_features_dims=True; user_custom_io=[]; disable_transform_tracking=False; disable_defer_loading=False; enable_Layout_Transform_v1=False; inject_cast_for_gather=True; converter_op_package_lib=; enable_experimental_feature=[]; handle_gather_negative_indices=True; disable_preserve_io=False; enable_tensor_deduplication=False; disable_batchnorm_folding=Fal

By following these steps, the model is optimized for efficient inference on Qualcomm® platforms, leveraging hardware acceleration for real-time AI applications. This process ensures that the model is both accurate and performant, making it suitable for deployment in edge devices powered by Qualcomm® chipsets.